# Construction des graphes

Dans cette partie nous allons mettre en place un graphe de citations et un graphe des auteurs pour étudier leurs caractéristiques et les implémenter dans notre moteur de recherche.

## Chargement des bilbiothèques nécessaires

In [1]:
import json
import numpy as np
import networkx as nx
from collections import defaultdict
from itertools import combinations
import os
import pickle
from sklearn.metrics import roc_auc_score

import warnings

warnings.filterwarnings("ignore")

# For embeddings and similarity computation
try:
    from sentence_transformers import SentenceTransformer
    from sklearn.metrics.pairwise import cosine_similarity

    print("Required libraries imported successfully!")
except ImportError as e:
    print(f"Missing library: {e}")
    print("Please install with: pip install sentence-transformers scikit-learn nx")

Required libraries imported successfully!


## Chargement des Corpus

In [2]:
def load_corpus(file_path: str) -> dict[str, dict]:
    """
    Load corpus data from JSONL file.
    Returns dictionary mapping document IDs to document data.
    """
    corpus = {}
    with open(file_path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            line = line.strip()
            obj = json.loads(line)
            docid = str(obj["_id"])
            corpus[docid] = obj
    return corpus


def load_queries(file_path: str) -> dict[str, dict]:
    """
    Load query data from JSONL file.
    Returns dictionary mapping query IDs to query data.
    """
    queries = {}
    with open(file_path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            line = line.strip()
            obj = json.loads(line)
            qid = str(obj["_id"])
            queries[qid] = obj
    return queries


def load_qrels(file_path: str) -> dict[str, dict[str, int]]:
    """
    Load relevance judgments from TSV file.
    Returns dictionary mapping query IDs to candidate relevance scores.
    """
    qrels = defaultdict(dict)
    with open(file_path, "r") as f:
        lines = f.readlines()
        for line in lines[1:]:
            qid, docid, score = line.strip().split("\t")
            qrels[qid][docid] = int(score)
    return qrels


def load_test_qrels(file_path: str) -> dict[str, dict[str, int]]:
    """
    Load relevance judgments from TSV file.
    Returns dictionary mapping query IDs to candidate relevance scores.
    """
    qrels = defaultdict(dict)
    with open(file_path, "r") as f:
        lines = f.readlines()
        for line in lines[1:]:
            qid, docid, _ = line.strip().split("\t")
            qrels[qid][docid] = 0
    return qrels


print("Loading dataset...")
corpus = load_corpus("./data/corpus.jsonl")
queries = load_queries("./data/queries.jsonl")
qrels_valid = load_qrels("./data/valid.tsv")
qrels_test = load_test_qrels("./data/test_final.tsv")


print(f"Loaded {len(corpus)} documents in corpus")
print(f"Loaded {len(queries)} queries")
print(f"Loaded relevance for {len(qrels_valid)} queries (dataset)")
print(f"Loaded relevance for {len(qrels_test)} queries (dataset)")

Loading dataset...
Loaded 25657 documents in corpus
Loaded 1000 queries
Loaded relevance for 700 queries (dataset)
Loaded relevance for 300 queries (dataset)


## Mise en place du graphe de citations

In [3]:
def create_corpus_graph_citation(corpus: dict[str, dict]) -> nx.DiGraph:
    """
    Create a graph from the corpus where each document is a node
    and edges represent citations between documents.
    """
    G = nx.DiGraph()
    for _id, docdata in corpus.items():
        G.add_node(_id, **docdata)
        metadata = docdata.get("metadata", {})
        citations = metadata.get("cited_by", [])
        for cited_docid in citations:
            if cited_docid in corpus:
                G.add_edge(_id, cited_docid)
    return G

In [4]:
corpus_graph_citation = create_corpus_graph_citation(corpus)
print(
    f"Corpus graph has {corpus_graph_citation.number_of_nodes()} nodes and {corpus_graph_citation.number_of_edges()} edges."
)

Corpus graph has 25657 nodes and 54005 edges.


### Statisques du graphe

In [5]:
num_nodes = corpus_graph_citation.number_of_nodes()
num_edges = corpus_graph_citation.number_of_edges()
print(f"\nNombre de nœuds: {num_nodes}")
print(f"Nombre d'arcs: {num_edges}")

# Densité du graphe
density = nx.density(corpus_graph_citation)
print(f"\nDensité du graphe: {density:.6f}")

# Degrés entrants et sortants
in_degrees = [
    corpus_graph_citation.in_degree(node) for node in corpus_graph_citation.nodes()
]
out_degrees = [
    corpus_graph_citation.out_degree(node) for node in corpus_graph_citation.nodes()
]

print("\n--- Degrés entrants (In-degree) ---")
print(f"Moyenne: {np.mean(in_degrees):.4f}")
print(f"Variance: {np.var(in_degrees):.4f}")
print(f"Écart-type: {np.std(in_degrees):.4f}")
print(f"Min: {np.min(in_degrees)}, Max: {np.max(in_degrees)}")

print("\n--- Degrés sortants (Out-degree) ---")
print(f"Moyenne: {np.mean(out_degrees):.4f}")
print(f"Variance: {np.var(out_degrees):.4f}")
print(f"Écart-type: {np.std(out_degrees):.4f}")
print(f"Min: {np.min(out_degrees)}, Max: {np.max(out_degrees)}")

print("\n" + "=" * 60)


Nombre de nœuds: 25657
Nombre d'arcs: 54005

Densité du graphe: 0.000082

--- Degrés entrants (In-degree) ---
Moyenne: 2.1049
Variance: 10.3450
Écart-type: 3.2164
Min: 0, Max: 83

--- Degrés sortants (Out-degree) ---
Moyenne: 2.1049
Variance: 138.6369
Écart-type: 11.7744
Min: 0, Max: 758



## Mise en place du graphe des auteurs

Pour ce graphe, deux oeuvres sont liées lorsque celles-ci ont un même auteur.

In [6]:
def create_corpus_author_graph(corpus: dict[str, dict]) -> nx.DiGraph:
    """
    Create a graph from the corpus where each document is a node
    and edges represent authorship between documents.
    """
    G = nx.DiGraph()
    for _id, docdata in corpus.items():
        G.add_node(_id, **docdata)

    author_to_docs: dict[str, set[str]] = defaultdict(set)
    for doc_id, docdata in corpus.items():
        metadata = docdata.get("metadata", {})
        authors = metadata.get("authors", []) or []
        normalized = []
        for a in authors:
            normalized.append(str(a))
        for name in set(normalized):
            author_to_docs[name].add(doc_id)

    for docs in author_to_docs.values():
        for doc1, doc2 in combinations(docs, 2):
            G.add_edge(doc1, doc2)
            G.add_edge(doc2, doc1)
    return G

In [7]:
corpus_author_graph = create_corpus_author_graph(corpus)
print(
    f"Corpus author graph has {corpus_author_graph.number_of_nodes()} nodes and {corpus_author_graph.number_of_edges()} edges."
)

Corpus author graph has 25657 nodes and 78406 edges.


### Statistiques du graphe des auteurs

In [8]:
num_nodes = corpus_author_graph.number_of_nodes()
num_edges = corpus_author_graph.number_of_edges()
print(f"\nNombre de nœuds: {num_nodes}")
print(f"Nombre d'arcs: {num_edges}")

# Densité du graphe
density = nx.density(corpus_author_graph)
print(f"\nDensité du graphe: {density:.6f}")

# Degrés entrants et sortants
in_degrees = [
    corpus_author_graph.in_degree(node) for node in corpus_author_graph.nodes()
]
out_degrees = [
    corpus_author_graph.out_degree(node) for node in corpus_author_graph.nodes()
]

print("\n--- Degrés entrants (In-degree) ---")
print(f"Moyenne: {np.mean(in_degrees):.4f}")
print(f"Variance: {np.var(in_degrees):.4f}")
print(f"Écart-type: {np.std(in_degrees):.4f}")
print(f"Min: {np.min(in_degrees)}, Max: {np.max(in_degrees)}")

print("\n--- Degrés sortants (Out-degree) ---")
print(f"Moyenne: {np.mean(out_degrees):.4f}")
print(f"Variance: {np.var(out_degrees):.4f}")
print(f"Écart-type: {np.std(out_degrees):.4f}")
print(f"Min: {np.min(out_degrees)}, Max: {np.max(out_degrees)}")

print("\n" + "=" * 60)


Nombre de nœuds: 25657
Nombre d'arcs: 78406

Densité du graphe: 0.000119

--- Degrés entrants (In-degree) ---
Moyenne: 3.0559
Variance: 36.3287
Écart-type: 6.0273
Min: 0, Max: 89

--- Degrés sortants (Out-degree) ---
Moyenne: 3.0559
Variance: 36.3287
Écart-type: 6.0273
Min: 0, Max: 89



Le graphe des auteurs est bien un graphe non-orienté comme les statistiques de degrées entrants et sortants sont les mêmes.

Le fait d'avoir une partie d'une base de données est pour les deux graphes assez discriminant car cela nous ampute arbitrairement de données (on ne sait pas sur quel critère a été fait la prise des données).

## Mise en place d'indicateurs de centralité

In [9]:
def pageRank_clustering(G: nx.DiGraph, alpha: float = 0.85) -> dict[str, float]:
    """
    Compute PageRank scores for each node in the graph G.
    Returns a dictionary mapping node IDs to their PageRank scores.
    """
    pagerank_scores = nx.pagerank(G, alpha=alpha)
    return pagerank_scores


def eigenvector_centrality_clustering(
    G: nx.DiGraph, max_iter: int = 1000, tol: float = 1.0e-6
) -> dict[str, float]:
    """
    Compute Eigenvector Centrality for each node in the graph G.
    Returns a dictionary mapping node IDs to their Eigenvector Centrality scores.
    """
    eigenvector_scores = nx.eigenvector_centrality(G, max_iter=max_iter, tol=tol)
    return eigenvector_scores


def clustering_coefficient_clustering(
    G: nx.DiGraph,
) -> dict[str, float]:
    """
    Compute Clustering Coefficient for each node in the graph G.
    Returns a dictionary mapping node IDs to their Clustering Coefficient scores.
    """
    clustering_scores = nx.clustering(G.to_undirected())
    return clustering_scores


def closeness_centrality_clustering(
    G: nx.DiGraph,
) -> dict[str, float]:
    """
    Compute Closeness Centrality for each node in the graph G.
    Returns a dictionary mapping node IDs to their Closeness Centrality scores.
    closeness_scores = nx.closeness_centrality(G)
    """
    closeness_scores = nx.closeness_centrality(G)
    return closeness_scores


def display_top_score(type_clustering: str, corpus_graph: nx.DiGraph, top_n: int = 10):
    """
    Display the top N nodes with the highest PageRank scores.
    """
    if type_clustering == "pagerank":
        scores = pageRank_clustering(corpus_graph, alpha=0.85)
    elif type_clustering == "eigenvector":
        scores = eigenvector_centrality_clustering(corpus_graph)
    elif type_clustering == "clustering":
        scores = clustering_coefficient_clustering(corpus_graph)
    elif type_clustering == "closeness":
        scores = closeness_centrality_clustering(corpus_graph)
    else:
        raise ValueError(
            "Invalid clustering type. Choose from 'pagerank', 'eigenvector', 'clustering'."
        )
    sorted_scores = sorted(scores.items(), key=lambda item: item[1], reverse=True)
    print(f"Top {top_n} documents by {type_clustering} score:")
    for rank, (docid, score) in enumerate(sorted_scores[:top_n], start=1):
        print(f"{rank}. Document ID: {docid}, {type_clustering} score: {score:.6f}")
    return sorted_scores

### Comparaison des centralités

In [10]:
def comparisons_centralities_scores(
    page_rank_scores,
    betweenness_scores,
    eigenvector_scores,
    top_k: int = 10,
):
    """
    Compare the top documents from PageRank and Betweenness Centrality scores.
    """
    page_rank_top_docs = {docid for docid, _ in page_rank_scores[:top_k]}
    betweenness_top_docs = {docid for docid, _ in betweenness_scores[:top_k]}
    eigenvecteor_top_docs = {docid for docid, _ in eigenvector_scores[:top_k]}

    common_docs = page_rank_top_docs.intersection(betweenness_top_docs)
    print(f"Documents in both top {top_k} PageRank and Betweenness Centrality:")
    for docid in common_docs:
        print(f"- Document ID: {docid}")

    common_docs_eigen = page_rank_top_docs.intersection(eigenvecteor_top_docs)
    print(f"\nDocuments in both top {top_k} PageRank and Eigenvector Centrality:")
    for docid in common_docs_eigen:
        print(f"- Document ID: {docid}")

    common_docs_all = page_rank_top_docs.intersection(
        betweenness_top_docs, eigenvecteor_top_docs
    )
    print(
        f"\nDocuments in top {top_k} of PageRank, Betweenness Centrality, and Eigenvector Centrality:"
    )
    for docid in common_docs_all:
        print(f"- Document ID: {docid}")

### Centralité du graphe des citations

In [11]:
pagerank_scores = display_top_score("pagerank", corpus_graph_citation, top_n=10)
eigenvector_scores = display_top_score("eigenvector", corpus_graph_citation, top_n=10)
clustering_scores = display_top_score("clustering", corpus_graph_citation, top_n=10)
closeness_scores = display_top_score("closeness", corpus_graph_citation, top_n=10)

comparisons_centralities_scores(
    pagerank_scores, closeness_scores, eigenvector_scores, 25
)

Top 10 documents by pagerank score:
1. Document ID: bf9db8ca2dce7386cbed1ae0fd6465148cdb2b98, pagerank score: 0.000535
2. Document ID: 4ea92f2ddbb7ce610c6e80377e61397bad7309ff, pagerank score: 0.000534
3. Document ID: 126df9f24e29feee6e49e135da102fbbd9154a48, pagerank score: 0.000456
4. Document ID: 9878cc39383560752c5379a7e9641fc82a4daf7f, pagerank score: 0.000354
5. Document ID: 27a918a368e971a15b545abf63353e269a8ce8a2, pagerank score: 0.000326
6. Document ID: 0c9c3f948eda8fb339e76c6612ca4bd36244efd7, pagerank score: 0.000310
7. Document ID: 9a7def005efb5b4984886c8a07ec4d80152602ab, pagerank score: 0.000298
8. Document ID: 42dfb714a9faddf3f7047d482b2e0531d884dc67, pagerank score: 0.000286
9. Document ID: 57c72cb88843d44b43192741c7010558bb451394, pagerank score: 0.000284
10. Document ID: 58f7accbe36aadd0ef83fd2746c879079eb816bc, pagerank score: 0.000284
Top 10 documents by eigenvector score:
1. Document ID: 383e3b9e408f57052bbbc430b8e6b60c0e31f7ef, eigenvector score: 0.247765
2. Docum

### Centralité du graphe des auteurs

In [11]:
pagerank_scores = display_top_score("pagerank", corpus_author_graph, top_n=10)
eigenvector_scores = display_top_score("eigenvector", corpus_author_graph, top_n=10)
clustering_scores = display_top_score("clustering", corpus_author_graph, top_n=10)
closeness_scores = display_top_score("closeness", corpus_author_graph, top_n=10)

comparisons_centralities_scores(
    pagerank_scores, closeness_scores, eigenvector_scores, 25
)

Top 10 documents by pagerank score:
1. Document ID: 8462ca8f2459bcf35378d6dbb10dce70a6fba70a, pagerank score: 0.000414
2. Document ID: a10b7a2c80e2bae49d46b980ed03c074fe36bb2a, pagerank score: 0.000333
3. Document ID: 5c3785bc4dc07d7e77deef7e90973bdeeea760a5, pagerank score: 0.000304
4. Document ID: 1c0fb6b1bbfde0f9bab6268f5609cce2bd3bc5bd, pagerank score: 0.000294
5. Document ID: 8e0eacf11a22b9705a262e908f17b1704fd21fa7, pagerank score: 0.000258
6. Document ID: 90614cea8c2ab2bff0343231a26d6d0c9315d6c7, pagerank score: 0.000258
7. Document ID: 480d545ac4a4ffff5b1bc291c2de613192e35d91, pagerank score: 0.000248
8. Document ID: 3b9484449d77317ca1cb6a6c44c50c99879a8f0e, pagerank score: 0.000248
9. Document ID: 1967ad3ac8a598adc6929e9e6b9682734f789427, pagerank score: 0.000241
10. Document ID: 0b98fd7bf704876e6d49eb5c9310b37b78989112, pagerank score: 0.000235
Top 10 documents by eigenvector score:
1. Document ID: 15cf63f8d44179423b4100531db4bb84245aa6f1, eigenvector score: 0.157399
2. Docum

On s'aperçoit que pour le graphe des auteurs, il y des noeuds avec une centralité bien plus forte que pour le graphe des citations. Il y a donc quelques auteurs revenant fortement par rapport aux autres, bien plus qu'une étude citée par d'autre citations.

## Embeddings avec les graphes

In [12]:
def get_node_embeddings_from_corpus(
    corpus_ids: list[str], corpus_emb: np.ndarray
) -> dict[str, np.ndarray]:
    """
    Create a mapping from node IDs to their embeddings.
    """
    embeddings = {}
    if isinstance(corpus_emb, dict):
        corpus_emb = corpus_emb.copy()
        corpus_emb = corpus_emb["corpus_emb"]

    for i, node_id in enumerate(corpus_ids):
        embeddings[node_id] = corpus_emb[i]
    return embeddings


def aggregate_neighbor_embeddings(
    G: nx.DiGraph, node_embeddings: dict[str, np.ndarray], weight: float = 0.5
) -> dict[str, np.ndarray]:
    """
    Improve node embeddings by aggregating embeddings from neighboring nodes.

    Args:
        G: nx directed graph
        node_embeddings: Dict mapping node IDs to their embedding vectors
        weight: Weight factor for neighbor embeddings (0 to 1)

    Returns:
        Dict mapping node IDs to improved embedding vectors
    """
    improved_embeddings = {}

    for node in G.nodes():
        if node not in node_embeddings:
            # Si le nœud n'a pas d'embedding, le garder tel quel
            improved_embeddings[node] = node_embeddings.get(node, np.zeros(384))
            continue

        original_emb = node_embeddings[node].copy()
        neighbor_embeddings = []

        for neighbor in G.successors(node):
            if neighbor in node_embeddings:
                neighbor_embeddings.append(node_embeddings[neighbor])

        for neighbor in G.predecessors(node):
            if neighbor in node_embeddings:
                neighbor_embeddings.append(node_embeddings[neighbor])

        if neighbor_embeddings:
            neighbor_agg = np.mean(neighbor_embeddings, axis=0)
            improved_embeddings[node] = (
                1 - weight
            ) * original_emb + weight * neighbor_agg
        else:
            improved_embeddings[node] = original_emb

    return improved_embeddings

In [13]:
def iterative_node_embedding_aggregation(
    G: nx.DiGraph,
    node_embeddings: dict[str, np.ndarray],
    iterations: int = 2,
    weight: float = 0.3,
) -> dict[str, np.ndarray]:
    """
    Iteratively improve node embeddings by aggregating neighbor embeddings.
    Multiple iterations allow information to propagate further in the graph.

    Args:
        G: nx directed graph
        node_embeddings: Initial node embeddings
        iterations: Number of aggregation iterations
        weight: Weight factor for neighbor embeddings

    Returns:
        Improved node embeddings after multiple iterations
    """
    current_embeddings = node_embeddings.copy()

    for _ in range(iterations):
        current_embeddings = aggregate_neighbor_embeddings(
            G, current_embeddings, weight=weight
        )

    return current_embeddings

### Tests des fonctions

In [51]:
encoder_path = "data/encoder_GIST-small-Embedding-v0.pkl"
if os.path.exists(encoder_path):
    with open(encoder_path, "rb") as f:
        encoder = pickle.load(f)
    corpus_ids = [_id for _id in corpus.keys()]
    node_embeddings = get_node_embeddings_from_corpus(corpus_ids, encoder)
    print(f"Loaded embeddings for {len(node_embeddings)} nodes")
else:
    print(f"Encoder file {encoder_path} not found. Please run the encoder first.")
    raise FileNotFoundError(f"Encoder file {encoder_path} not found.")

improved_embeddings = aggregate_neighbor_embeddings(
    corpus_graph_citation, node_embeddings, weight=0.5
)

improved_embeddings_multi = iterative_node_embedding_aggregation(
    corpus_graph_citation, node_embeddings, iterations=3, weight=0.3
)
print(f"Improved {len(improved_embeddings_multi)} node embeddings")

Loaded embeddings for 25657 nodes
Improved 25657 node embeddings


## Evaluation des modèles

### Mise en place de notre moteur de recherche

In [31]:
def search_engine(
    query: str,
    corpus: dict[str, dict],
    embeddings: np.ndarray,
    model: SentenceTransformer,
    top_k: int = 5,
    all=False,
) -> list[tuple[str, float]]:
    """
    A search engine that retrieves the top_k most similar documents to the query
    based on cosine similarity of sentence embeddings.
    """
    query_embedding = model.encode([query], convert_to_numpy=True)
    similarities = cosine_similarity(query_embedding, embeddings)[0]
    if all:
        all_indices = np.argsort(similarities)[::-1]
        results = [(list(corpus.keys())[i], similarities[i]) for i in all_indices]
        return results
    else:
        top_indices = np.argsort(similarities)[-top_k:][::-1]
        results = [(list(corpus.keys())[i], similarities[i]) for i in top_indices]
        return results

### Mise en place des différents modèles denses disponibles

In [15]:
def loader_model(model_name: str) -> tuple[SentenceTransformer, np.ndarray]:
    if model_name == "intfloat/e5-small-v2" or model_name == "e5-small-v2":
        model = SentenceTransformer("intfloat/e5-small-v2")
        sentence_embeddings = pickle.load(open("./data/encoder_e5-small-v2.pkl", "rb"))[
            "corpus_emb"
        ]
    elif model_name == "all-MiniLM-L6-v2":
        model = SentenceTransformer("all-MiniLM-L6-v2")
        sentence_embeddings = pickle.load(
            open("./data/sentence_embeddings_all-MiniLM-L6-v2.pkl", "rb")
        )
    elif (
        model_name == "GIST-small-Embedding-v0"
        or model_name == "avsolatorio/GIST-small-Embedding-v0"
    ):
        model = SentenceTransformer("avsolatorio/GIST-small-Embedding-v0")
        sentence_embeddings = pickle.load(
            open("./data/encoder_GIST-small-Embedding-v0.pkl", "rb")
        )
    elif (
        model_name == "multilingual-e5-small"
        or model_name == "intfloat/multilingual-e5-small"
    ):
        model = SentenceTransformer("intfloat/multilingual-e5-small")
        sentence_embeddings = pickle.load(
            open("./data/encoder_multilingual-e5-small.pkl", "rb")
        )
    else:
        raise ValueError(f"Model {model_name} not supported.")
    return model, sentence_embeddings

### Mise en place du modèle et de ses statistiques

In [21]:
def evaluate_model(
    qrels_valid: dict[str, dict[str, int]],
    queries: dict[str, dict],
    corpus: dict[str, dict],
    model_name: str,
    graph: list[nx.DiGraph],
    top_k: int = 5,
    weight: float = 0.3,
):
    model, sentence_embeddings = loader_model(model_name)

    corpus_keys = list(corpus.keys())
    corpus_index = {doc_id: idx for idx, doc_id in enumerate(corpus_keys)}

    embeddings = get_node_embeddings_from_corpus(corpus_keys, sentence_embeddings)

    if len(graph) == 0:
        raise ValueError("Graph list is empty.")

    for g in graph:
        embeddings = iterative_node_embedding_aggregation(
            g, embeddings, iterations=3, weight=weight
        )

    # Correction des embeddings
    if isinstance(embeddings, dict):
        if "corpus_emb" in embeddings:
            corpus_emb = embeddings["corpus_emb"]
            get_emb = lambda doc_id: corpus_emb[corpus_index[doc_id]]
        else:
            get_emb = lambda doc_id: embeddings[doc_id]
    else:
        get_emb = lambda doc_id: embeddings[corpus_index[doc_id]]

    precisions: list[float] = []
    rappels: list[float] = []
    mrrs: list[float] = []
    ndcgs: list[float] = []
    maps: list[float] = []
    aucs: list[float] = []
    AUC_data = {}

    all_results = {}

    for qid, labels in qrels_valid.items():
        relevant_ids = {
            doc_id for doc_id, rel in labels.items() if rel == 1 and doc_id in corpus
        }
        if not relevant_ids:
            continue  # Skip queries without any relevant doc in corpus

        restricted_corpus = {
            doc_id: corpus[doc_id] for doc_id in labels if doc_id in corpus
        }
        corpus_list = list(restricted_corpus.keys())
        try:
            restricted_embeddings = np.array(
                [get_emb(doc_id) for doc_id in corpus_list]
            )
        except KeyError as e:
            print(f"Skipping query {qid} due to missing embedding for doc {e}")
            continue

        # Compute similarities for AUC across the whole restricted corpus
        query_embedding = model.encode([queries[qid]["text"]], convert_to_numpy=True)
        full_similarities = cosine_similarity(query_embedding, restricted_embeddings)[0]
        y_true = [labels[doc_id] for doc_id in corpus_list]
        try:
            auc = roc_auc_score(y_true, full_similarities)
            aucs.append(auc)
        except ValueError:
            # AUC undefined if only one class is present; skip this query
            pass

        results = search_engine(
            queries[qid]["text"],
            restricted_corpus,
            restricted_embeddings,
            model,
            top_k=top_k,
        )
        all_results[qid] = results

        hits: list[int] = []
        dcg = 0.0
        mrr = 0.0
        cum_hits = 0
        ap_acc = 0.0

        for rank, (doc_id, _) in enumerate(results):
            hit = 1 if doc_id in relevant_ids else 0
            hits.append(hit)
            cum_hits += hit
            if hit and mrr == 0.0:
                mrr = 1.0 / (rank + 1)
            if hit:
                ap_acc += cum_hits / (rank + 1)
            dcg += hit / np.log2(rank + 2)

        ideal_rel = min(len(relevant_ids), top_k)
        idcg = sum(1 / np.log2(i + 2) for i in range(ideal_rel))
        ndcg = dcg / idcg if idcg > 0 else 0.0

        precision = sum(hits) / top_k
        rappel = sum(hits) / len(relevant_ids)
        ap = ap_acc / len(relevant_ids)

        precisions.append(precision)
        rappels.append(rappel)
        mrrs.append(mrr)
        ndcgs.append(ndcg)
        maps.append(ap)

        # AUC data collection
        true_positives = sum(hits)
        true_negatives = (
            len(restricted_corpus) - len(relevant_ids) - (top_k - true_positives)
        )

        if true_positives not in AUC_data:
            AUC_data[true_positives] = {}
        if true_negatives not in AUC_data[true_positives]:
            AUC_data[true_positives][true_negatives] = 0
        AUC_data[true_positives][true_negatives] += 1

    print(f"Model: {model_name}")
    print(f"Precision@{top_k}: {np.mean(precisions):.4f}")
    print(f"rappel@{top_k}: {np.mean(rappels):.4f}")
    print(
        f"F1@{top_k}: {2 * np.mean(precisions) * np.mean(rappels) / (np.mean(precisions) + np.mean(rappels) + 1e-10):.4f}"
    )
    print(f"MRR@{top_k}: {np.mean(mrrs):.4f}")
    print(f"nDCG@{top_k}: {np.mean(ndcgs):.4f}")
    print(f"MAP@{top_k}: {np.mean(maps):.4f}")
    if aucs:
        print(f"AUC: {np.mean(aucs):.4f}")
    else:
        print("AUC: N/A (need queries with both positive and negative labels)")
    return all_results

### Tests avec les graphes

In [19]:
print("\n\nEvaluating model GIST-small-Embedding-v0 with citation graph")
evaluate_model(
    qrels_valid,
    queries,
    corpus,
    "GIST-small-Embedding-v0",
    graph=[corpus_graph_citation],
    top_k=5,
)

print("\n\nEvaluating model GIST-small-Embedding-v0 with author graph")

evaluate_model(
    qrels_valid,
    queries,
    corpus,
    "GIST-small-Embedding-v0",
    graph=[corpus_author_graph],
    top_k=5,
)

print("\n\nEvaluating model GIST-small-Embedding-v0 without both graphs")
evaluate_model(
    qrels_valid,
    queries,
    corpus,
    "GIST-small-Embedding-v0",
    graph=[corpus_author_graph, corpus_graph_citation],
    top_k=5,
)
print("\n\nTesting model GIST-small-Embedding-v0 on test set")



Evaluating model GIST-small-Embedding-v0 with citation graph
Model: GIST-small-Embedding-v0
Precision@5: 0.8540
rappel@5: 0.8670
F1@5: 0.8604
MRR@5: 0.9808
nDCG@5: 0.8939
MAP@5: 0.8422
AUC: 0.9769


Evaluating model GIST-small-Embedding-v0 with author graph
Model: GIST-small-Embedding-v0
Precision@5: 0.7960
rappel@5: 0.8076
F1@5: 0.8018
MRR@5: 0.9768
nDCG@5: 0.8491
MAP@5: 0.7784
AUC: 0.9563


Evaluating model GIST-small-Embedding-v0 without both graphs
Model: GIST-small-Embedding-v0
Precision@5: 0.8497
rappel@5: 0.8627
F1@5: 0.8562
MRR@5: 0.9801
nDCG@5: 0.8907
MAP@5: 0.8386
AUC: 0.9751


Testing model GIST-small-Embedding-v0 on test set


### Etude de l'effet du poid

In [25]:
for i in range(1, 10):
    print(
        f"\n\nEvaluating model GIST-small-Embedding-v0 with citation graph with weight : {i / 10}"
    )
    evaluate_model(
        qrels_valid,
        queries,
        corpus,
        "GIST-small-Embedding-v0",
        graph=[corpus_graph_citation],
        top_k=5,
        weight=0.3 + i / 100,
    )

print("\n\nTesting model GIST-small-Embedding-v0 on test set")



Evaluating model GIST-small-Embedding-v0 with citation graph with weight : 0.1
Model: GIST-small-Embedding-v0
Precision@5: 0.8551
rappel@5: 0.8681
F1@5: 0.8616
MRR@5: 0.9815
nDCG@5: 0.8948
MAP@5: 0.8433
AUC: 0.9770


Evaluating model GIST-small-Embedding-v0 with citation graph with weight : 0.2
Model: GIST-small-Embedding-v0
Precision@5: 0.8546
rappel@5: 0.8675
F1@5: 0.8610
MRR@5: 0.9822
nDCG@5: 0.8946
MAP@5: 0.8429
AUC: 0.9771


Evaluating model GIST-small-Embedding-v0 with citation graph with weight : 0.3
Model: GIST-small-Embedding-v0
Precision@5: 0.8546
rappel@5: 0.8675
F1@5: 0.8610
MRR@5: 0.9808
nDCG@5: 0.8943
MAP@5: 0.8427
AUC: 0.9772


Evaluating model GIST-small-Embedding-v0 with citation graph with weight : 0.4
Model: GIST-small-Embedding-v0
Precision@5: 0.8549
rappel@5: 0.8677
F1@5: 0.8612
MRR@5: 0.9815
nDCG@5: 0.8945
MAP@5: 0.8432
AUC: 0.9773


Evaluating model GIST-small-Embedding-v0 with citation graph with weight : 0.5
Model: GIST-small-Embedding-v0
Precision@5: 0.8546


## Exportation des résultats

In [34]:
def test_model(
    qrels_test: dict[str, dict[str, int]],
    queries: dict[str, dict],
    corpus: dict[str, dict],
    model_name: str,
    graph: nx.DiGraph,
    top_k: int = 5,
    weight: float = 0.3,
):
    model, sentence_embeddings = loader_model(model_name)
    qrels_results = qrels_test.copy()
    corpus_keys = list(corpus.keys())
    corpus_index = {doc_id: idx for idx, doc_id in enumerate(corpus_keys)}

    node_embeddings = get_node_embeddings_from_corpus(corpus_keys, sentence_embeddings)
    embeddings = iterative_node_embedding_aggregation(
        graph, node_embeddings, iterations=3, weight=weight
    )

    # Correction des embeddings
    if isinstance(embeddings, dict):
        if "corpus_emb" in embeddings:
            corpus_emb = embeddings["corpus_emb"]
            get_emb = lambda doc_id: corpus_emb[corpus_index[doc_id]]
        else:
            get_emb = lambda doc_id: embeddings[doc_id]
    else:
        get_emb = lambda doc_id: embeddings[corpus_index[doc_id]]

    for qid, labels in qrels_test.items():
        restricted_corpus = {
            doc_id: corpus[doc_id] for doc_id in labels if doc_id in corpus
        }
        corpus_list = list(restricted_corpus.keys())

        try:
            restricted_embeddings = np.array(
                [get_emb(doc_id) for doc_id in corpus_list]
            )
        except KeyError as e:
            print(f"Skipping query {qid} due to missing embedding for doc {e}")
            continue

        results = search_engine(
            queries[qid]["text"],
            restricted_corpus,
            restricted_embeddings,
            model,
            top_k=top_k,
            all=True,
        )
        qrels_results[qid] = {doc_id: float(score) for doc_id, score in results}
    return qrels_results

In [28]:
def export_results(qrels_results: dict[str, dict[str, int]], output_file: str):
    """
    Export the test results to a csv file.
    """
    with open(output_file, "w", encoding="utf-8") as f:
        f.write("RowId,query-id,corpus-id,score\n")
        count = 1
        for qid, doc_scores in qrels_results.items():
            for doc_id, score in doc_scores.items():
                f.write(f"{count},{qid},{doc_id},{score}\n")
                count += 1
    print(f"Results exported to {output_file}")

In [35]:
model_name = "GIST-small-Embedding-v0"

results = test_model(
    qrels_test,
    queries,
    corpus,
    model_name=model_name,
    graph=corpus_graph_citation,
    top_k=5,
    weight=0.34,
)
export_results(results, f"./results/test_results_{model_name}_node.csv")


Results exported to ./results/test_results_GIST-small-Embedding-v0_node.csv


Evaluating model GIST-small-Embedding-v0 with citation graph with weight : 0.2
Model: GIST-small-Embedding-v0
Precision@5: 0.8477
rappel@5: 0.8604
F1@5: 0.8540
MRR@5: 0.9831
nDCG@5: 0.8892
MAP@5: 0.8349
AUC: 0.9758


Evaluating model GIST-small-Embedding-v0 with citation graph with weight : 0.3
Model: GIST-small-Embedding-v0
Precision@5: 0.8540
rappel@5: 0.8670
F1@5: 0.8604
MRR@5: 0.9808
nDCG@5: 0.8939
MAP@5: 0.8422
AUC: 0.9769


Evaluating model GIST-small-Embedding-v0 with citation graph with weight : 0.4
Model: GIST-small-Embedding-v0
Precision@5: 0.8531
rappel@5: 0.8660
F1@5: 0.8595
MRR@5: 0.9812
nDCG@5: 0.8929
MAP@5: 0.8412
AUC: 0.9769


Evaluating model GIST-small-Embedding-v0 with citation graph with weight : 0.5
Model: GIST-small-Embedding-v0
Precision@5: 0.8506
rappel@5: 0.8634
F1@5: 0.8569
MRR@5: 0.9810
nDCG@5: 0.8905
MAP@5: 0.8376
AUC: 0.9761